In [0]:
# 1 — Read Silver table
silver_df = spark.table("weather_silver_clean")

print(f"Total Silver rows available: {silver_df.count()}")
silver_df.show(5, truncate=False)

In [0]:
# 2 — Daily aggregations per city (the Gold table)
from pyspark.sql.functions import (
    avg, max as spark_max, min as spark_min,
    round as spark_round, count, date_trunc,
    first, col
)

gold_daily_df = silver_df \
    .withColumn("date", date_trunc("day", col("recorded_at"))) \
    .groupBy("city", "date", "latitude", "longitude", "country") \
    .agg(
        # Temperature stats
        spark_round(avg("temp_celsius"),        1).alias("avg_temp_celsius"),
        spark_round(spark_max("temp_celsius"),  1).alias("max_temp_celsius"),
        spark_round(spark_min("temp_celsius"),  1).alias("min_temp_celsius"),
        spark_round(avg("feels_like_celsius"),  1).alias("avg_feels_like"),

        # Atmosphere
        spark_round(avg("humidity_pct"),        1).alias("avg_humidity_pct"),
        spark_round(avg("pressure_hpa"),        1).alias("avg_pressure_hpa"),
        spark_round(avg("cloud_cover_pct"),     1).alias("avg_cloud_cover_pct"),

        # Wind
        spark_round(avg("wind_speed_mps"),      2).alias("avg_wind_speed_mps"),
        spark_round(spark_max("wind_speed_mps"),2).alias("max_wind_speed_mps"),

        # Most frequent condition of the day
        first("weather_main").alias("dominant_weather_main"),
        first("weather_description").alias("dominant_weather_description"),

        # Pipeline metadata
        count("*").alias("readings_count"),
        spark_max("recorded_at").alias("last_reading_at"),
    ) \
    .orderBy("date", "city")

print(f"✓ Gold daily aggregations computed: {gold_daily_df.count()} rows")
gold_daily_df.show(truncate=False)

In [0]:
# 3 — Write to Gold using MERGE
# Key: city + date — one summary row per city per day
# Re-running updates the row as more readings come in throughout the day
from delta.tables import DeltaTable

GOLD_TABLE = "weather_gold_daily"

if not spark.catalog.tableExists(GOLD_TABLE):
    gold_daily_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(GOLD_TABLE)
    print(f"✓ Gold table created — {gold_daily_df.count()} rows")

else:
    gold_table = DeltaTable.forName(spark, GOLD_TABLE)

    gold_table.alias("existing").merge(
        gold_daily_df.alias("new"),
        """
        existing.city = new.city
        AND existing.date = new.date
        """
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

    print(f"✓ Gold MERGE complete — {spark.table(GOLD_TABLE).count()} total rows")

In [0]:
# 4 — Verify Gold output
spark.sql(f"""
    SELECT
        city,
        DATE(date)              AS date,
        avg_temp_celsius,
        max_temp_celsius,
        min_temp_celsius,
        avg_humidity_pct,
        avg_wind_speed_mps,
        dominant_weather_description,
        readings_count,
        last_reading_at
    FROM {GOLD_TABLE}
    ORDER BY date DESC, avg_temp_celsius DESC
""").show(truncate=False)

In [0]:
# 5 — Analytical queries on Gold
# These are the kind of questions stakeholders/HRs would ask

print("=" * 60)
print("🌡  HOTTEST CITY TODAY")
print("=" * 60)
spark.sql(f"""
    SELECT city, avg_temp_celsius, dominant_weather_description
    FROM {GOLD_TABLE}
    WHERE DATE(date) = CURRENT_DATE()
    ORDER BY avg_temp_celsius DESC
    LIMIT 1
""").show(truncate=False)

print("=" * 60)
print("💧 MOST HUMID CITY TODAY")
print("=" * 60)
spark.sql(f"""
    SELECT city, avg_humidity_pct, avg_temp_celsius
    FROM {GOLD_TABLE}
    WHERE DATE(date) = CURRENT_DATE()
    ORDER BY avg_humidity_pct DESC
    LIMIT 1
""").show(truncate=False)

print("=" * 60)
print("🌬  WINDIEST CITY TODAY")
print("=" * 60)
spark.sql(f"""
    SELECT city, avg_wind_speed_mps, max_wind_speed_mps, dominant_weather_description
    FROM {GOLD_TABLE}
    WHERE DATE(date) = CURRENT_DATE()
    ORDER BY avg_wind_speed_mps DESC
    LIMIT 1
""").show(truncate=False)

print("=" * 60)
print("📊 FULL CITY RANKING BY TEMPERATURE")
print("=" * 60)
spark.sql(f"""
    SELECT
        RANK() OVER (ORDER BY avg_temp_celsius DESC) AS rank,
        city,
        avg_temp_celsius,
        avg_humidity_pct,
        dominant_weather_description,
        readings_count
    FROM {GOLD_TABLE}
    WHERE DATE(date) = CURRENT_DATE()
""").show(truncate=False)